# Assessment 02 — SQL (via SQLite)

| | |
|---|---|
| **Estimated time** | 20–30 minutes (self-timed) |
| **Skill level tested** | SQL: SELECT → JOINs → CTEs → Window Functions → Subqueries |
| **Prerequisites** | `sqlite3` (stdlib) — no external DB or driver required |

---

## Instructions

All queries run against an **in-memory SQLite database** created in the setup
cell below.  The schema is intentionally SAP-like:

- `materials` — material master (≈ MM60 / MARA fields)
- `sales_orders` — order line items (≈ VA05 extract)
- `cost_centers` — simple cost-center dimension (≈ KS13)

**For each exercise:**
1. Read the problem statement.
2. Fill in the `query = """...(YOUR SQL HERE)..."""` placeholder.
3. Run the cell — results print automatically.

> Write standard SQL.  SQLite supports CTEs and window functions
> (RANK, ROW_NUMBER, LAG, LEAD) since version 3.25 (Python 3.6+).

Run the **Setup** cell first — it creates and seeds all three tables.


In [ ]:
%%time
import sqlite3
import pandas as pd

# ── Create in-memory database ─────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row   # allow column-name access
cur = conn.cursor()

# ── DDL ───────────────────────────────────────────────────────────────────────
cur.executescript("""
-- Materials master
CREATE TABLE materials (
    matnr           TEXT PRIMARY KEY,
    mtart           TEXT,        -- material type: ROH, HALB, FERT, NLAG
    maktx           TEXT,        -- description
    meins           TEXT,        -- base unit
    werks           TEXT,        -- plant
    labst           REAL,        -- unrestricted stock qty
    unit_price      REAL         -- valuation price
);

-- Sales orders (line items)
CREATE TABLE sales_orders (
    vbeln           TEXT,        -- sales doc number
    posnr           INTEGER,     -- line item
    matnr           TEXT,        -- material
    kunnr           TEXT,        -- customer number
    region          TEXT,        -- sales region
    order_date      TEXT,        -- ISO date
    status          TEXT,        -- OPEN | DELIVERED | CANCELLED
    qty_ordered     REAL,
    net_price       REAL         -- net unit price at time of order
);

-- Cost centers
CREATE TABLE cost_centers (
    kostl           TEXT PRIMARY KEY,
    kokrs           TEXT,        -- controlling area
    ktext           TEXT,        -- description
    bukrs           TEXT         -- company code
);
""")

# ── Seed: materials (12 rows) ─────────────────────────────────────────────────
materials_data = [
    ("100-100","ROH","Steel sheet 2mm","KG","1000",1200,12.50),
    ("100-101","ROH","Copper wire 1mm","M", "1000", 800, 8.75),
    ("100-102","HALB","Sub-assembly A",  "PC","1000", 350,45.00),
    ("100-103","HALB","Sub-assembly B",  "PC","1000", 220,62.30),
    ("100-104","FERT","Finished unit X", "PC","1000",  90,125.00),
    ("100-105","FERT","Finished unit Y", "PC","2000",  60,135.50),
    ("100-106","ROH","Aluminium rod",    "KG","2000", 950, 9.20),
    ("100-107","NLAG","Packaging foam",  "PC","1000",3000,  2.10),
    ("100-108","NLAG","Cardboard box",   "PC","2000",2100,  3.30),
    ("100-109","FERT","Finished unit Z", "PC","2000",  30,198.00),
    ("100-110","ROH","Rubber seal",      "PC","1000", 600,  5.60),
    ("100-111","HALB","Sub-assembly C",  "PC","2000", 180, 78.00),
]
cur.executemany(
    "INSERT INTO materials VALUES (?,?,?,?,?,?,?)", materials_data
)

# ── Seed: sales_orders (30 rows) ─────────────────────────────────────────────
sales_data = [
    # (vbeln, posnr, matnr, kunnr, region, order_date, status, qty, price)
    ("1000000001",10,"100-104","CUST-01","NORTH","2024-01-15","DELIVERED",20,128.00),
    ("1000000001",20,"100-105","CUST-01","NORTH","2024-01-15","DELIVERED",10,138.00),
    ("1000000002",10,"100-104","CUST-02","SOUTH","2024-02-10","DELIVERED",15,125.00),
    ("1000000002",20,"100-109","CUST-02","SOUTH","2024-02-10","DELIVERED", 8,200.00),
    ("1000000003",10,"100-102","CUST-03","EAST", "2024-03-05","DELIVERED",30, 46.00),
    ("1000000003",20,"100-103","CUST-03","EAST", "2024-03-05","CANCELLED",10, 65.00),
    ("1000000004",10,"100-105","CUST-01","NORTH","2024-04-20","DELIVERED",25,140.00),
    ("1000000004",20,"100-111","CUST-01","NORTH","2024-04-20","DELIVERED",12, 80.00),
    ("1000000005",10,"100-104","CUST-04","WEST", "2024-05-12","DELIVERED",18,130.00),
    ("1000000005",20,"100-109","CUST-04","WEST", "2024-05-12","OPEN",      6,198.00),
    ("1000000006",10,"100-102","CUST-02","SOUTH","2024-06-01","DELIVERED",40, 47.00),
    ("1000000006",20,"100-103","CUST-02","SOUTH","2024-06-01","DELIVERED",20, 63.00),
    ("1000000007",10,"100-105","CUST-05","EAST", "2024-07-18","DELIVERED",30,137.00),
    ("1000000007",20,"100-104","CUST-05","EAST", "2024-07-18","CANCELLED",  5,126.00),
    ("1000000008",10,"100-109","CUST-03","EAST", "2024-08-09","DELIVERED",12,202.00),
    ("1000000008",20,"100-111","CUST-03","EAST", "2024-08-09","DELIVERED",20, 79.00),
    ("1000000009",10,"100-104","CUST-06","NORTH","2024-09-22","DELIVERED",22,129.00),
    ("1000000009",20,"100-105","CUST-06","NORTH","2024-09-22","DELIVERED",18,139.00),
    ("1000000010",10,"100-102","CUST-04","WEST", "2024-10-30","DELIVERED",35, 45.00),
    ("1000000010",20,"100-109","CUST-04","WEST", "2024-10-30","OPEN",     10,199.00),
    ("1000000011",10,"100-104","CUST-01","NORTH","2024-11-14","DELIVERED",28,131.00),
    ("1000000011",20,"100-111","CUST-01","NORTH","2024-11-14","DELIVERED",15, 81.00),
    ("1000000012",10,"100-105","CUST-02","SOUTH","2024-12-05","DELIVERED",20,136.00),
    ("1000000012",20,"100-109","CUST-02","SOUTH","2024-12-05","DELIVERED",14,201.00),
    # 2025 orders
    ("1000000013",10,"100-104","CUST-01","NORTH","2025-01-10","DELIVERED",30,132.00),
    ("1000000013",20,"100-102","CUST-01","NORTH","2025-01-10","OPEN",     20, 46.00),
    ("1000000014",10,"100-109","CUST-03","EAST", "2025-02-14","DELIVERED",16,205.00),
    ("1000000014",20,"100-105","CUST-03","EAST", "2025-02-14","OPEN",     22,141.00),
    ("1000000015",10,"100-104","CUST-05","EAST", "2025-03-01","OPEN",     12,130.00),
    ("1000000015",20,"100-111","CUST-07","WEST", "2025-03-01","OPEN",     18, 82.00),
]
cur.executemany(
    "INSERT INTO sales_orders VALUES (?,?,?,?,?,?,?,?,?)", sales_data
)

# ── Seed: cost_centers (6 rows) ──────────────────────────────────────────────
cc_data = [
    ("CC-1000","CO01","Production dept A","1000"),
    ("CC-1001","CO01","Production dept B","1000"),
    ("CC-1002","CO01","Warehouse","1000"),
    ("CC-2000","CO02","Production dept A","2000"),
    ("CC-2001","CO02","Quality control","2000"),
    ("CC-9999","CO01","Overhead allocation","1000"),
]
cur.executemany("INSERT INTO cost_centers VALUES (?,?,?,?)", cc_data)
conn.commit()

# ── Helper: run query → DataFrame + print ─────────────────────────────────────
def run_query(query, label="Result"):
    df = pd.read_sql_query(query, conn)
    print(f"\n{'─'*60}")
    print(f"  {label}  ({len(df)} rows)")
    print('─'*60)
    print(df.to_string(index=False))
    return df

print("✓ Database seeded:")
print(f"  materials    : {cur.execute('SELECT COUNT(*) FROM materials').fetchone()[0]} rows")
print(f"  sales_orders : {cur.execute('SELECT COUNT(*) FROM sales_orders').fetchone()[0]} rows")
print(f"  cost_centers : {cur.execute('SELECT COUNT(*) FROM cost_centers').fetchone()[0]} rows")


---

## Exercise 1 — Top Materials by Inventory Value (Basic SELECT)

**Task:** Find the **top 10 materials** ranked by inventory value
(`labst × unit_price`), but only include materials with **unrestricted stock
greater than 0** (`labst > 0`).

**Required columns in output:**
`matnr`, `maktx`, `werks`, `labst`, `unit_price`, `inv_value`

Sorted by `inv_value` descending.

*Tests: column aliasing, arithmetic in SELECT, WHERE, ORDER BY, LIMIT.*


In [ ]:
%%time

query = """
-- YOUR SQL HERE
-- Hint: calculate inv_value as labst * unit_price
-- Filter: labst > 0
-- Sort: inv_value DESC
-- Limit: 10 rows
SELECT 1 AS placeholder  -- remove this line and write your query
"""

result1 = run_query(query, "Top 10 Materials by Inventory Value")


<details>
<summary>💡 Hint (click to expand)</summary>

```sql
SELECT
    matnr,
    maktx,
    werks,
    labst,
    unit_price,
    ROUND(labst * unit_price, 2) AS inv_value
FROM materials
WHERE labst > 0
ORDER BY inv_value DESC
LIMIT 10;
```

</details>


---

## Exercise 2 — Revenue & Order Count per Customer (JOIN + GROUP BY)

**Task:** Calculate **total revenue** and **order count** per customer, but
only for **DELIVERED** orders.

Revenue for a line = `qty_ordered × net_price`.

**Required columns:** `kunnr`, `total_revenue` (rounded to 2 dp),
`order_line_count`  
Sorted by `total_revenue` descending.

*Tests: INNER JOIN, GROUP BY, aggregate functions, WHERE on joined data.*


In [ ]:
%%time

query = """
-- YOUR SQL HERE
-- Tables: sales_orders (no join needed for this one — all data is in one table)
-- Filter: status = 'DELIVERED'
-- Aggregate: SUM(qty_ordered * net_price), COUNT(*)
SELECT 1 AS placeholder
"""

result2 = run_query(query, "Revenue & Order Count per Customer (DELIVERED only)")


<details>
<summary>💡 Hint (click to expand)</summary>

```sql
SELECT
    kunnr,
    ROUND(SUM(qty_ordered * net_price), 2) AS total_revenue,
    COUNT(*)                               AS order_line_count
FROM sales_orders
WHERE status = 'DELIVERED'
GROUP BY kunnr
ORDER BY total_revenue DESC;
```

This is a single-table aggregation — no join required.  A JOIN exercise would
come in if you needed to enrich with customer master data (imagine a `customers`
table with `kunnr` as a key).

</details>


---

## Exercise 3 — Above-Average Months (CTE)

**Task:**  
Using a CTE, first calculate **monthly revenue** (DELIVERED orders only, all
months in the data).  Then, in the outer query, return only the months where
revenue **exceeded the overall average monthly revenue**.

**Required columns:** `month` (format `YYYY-MM`), `monthly_revenue`,
`avg_monthly_revenue`  
Sorted by `month` ascending.

*Tests: `WITH` clause, subquery logic, `AVG()` as a window or scalar subquery.*


In [ ]:
%%time

query = """
-- YOUR SQL HERE
-- Step 1 (CTE): group by month, sum revenue for DELIVERED orders
-- Step 2 (outer): filter where monthly_revenue > overall avg
-- Hint: STRFTIME('%Y-%m', order_date) extracts the YYYY-MM string in SQLite

WITH monthly_revenue AS (
    -- YOUR CTE BODY HERE
    SELECT 'placeholder' AS month, 0.0 AS monthly_revenue
)
SELECT
    month,
    ROUND(monthly_revenue, 2)            AS monthly_revenue,
    ROUND(AVG(monthly_revenue) OVER (), 2) AS avg_monthly_revenue
FROM monthly_revenue
WHERE 1=0  -- replace this condition
ORDER BY month;
"""

result3 = run_query(query, "Months Exceeding Average Monthly Revenue")


<details>
<summary>💡 Hint (click to expand)</summary>

```sql
WITH monthly_revenue AS (
    SELECT
        STRFTIME('%Y-%m', order_date)   AS month,
        SUM(qty_ordered * net_price)    AS monthly_revenue
    FROM sales_orders
    WHERE status = 'DELIVERED'
    GROUP BY month
)
SELECT
    month,
    ROUND(monthly_revenue, 2)             AS monthly_revenue,
    ROUND(AVG(monthly_revenue) OVER (), 2) AS avg_monthly_revenue
FROM monthly_revenue
WHERE monthly_revenue > (SELECT AVG(monthly_revenue) FROM monthly_revenue)
ORDER BY month;
```

`AVG(...) OVER ()` is a window function that repeats the same scalar across
every row — a clean way to display the benchmark alongside each row.

</details>


---

## Exercise 4 — Rank Materials by Inventory Value per Plant (Window Function)

**Task:**  
Using `RANK() OVER (PARTITION BY ... ORDER BY ...)`, rank each material within
its plant (`werks`) by inventory value (`labst × unit_price`).

Return **only ranks 1, 2, and 3** per plant.

**Required columns:** `werks`, `rank_in_plant`, `matnr`, `maktx`,
`inv_value`  
Sorted by `werks` asc, then `rank_in_plant` asc.

> This pattern — ranking within groups — is one of the most common analytics
> SQL patterns in SAP reporting (e.g., top materials per plant/cost center).

*Tests: `RANK()`, `PARTITION BY`, `ORDER BY` inside `OVER()`, filtering on
window results via a subquery or CTE.*


In [ ]:
%%time

query = """
-- YOUR SQL HERE
-- You cannot filter on a window function alias directly in WHERE —
-- wrap in a subquery or CTE first.
-- Hint:
--   RANK() OVER (PARTITION BY werks ORDER BY labst * unit_price DESC)
SELECT 1 AS placeholder
"""

result4 = run_query(query, "Top 3 Materials per Plant by Inventory Value")


<details>
<summary>💡 Hint (click to expand)</summary>

```sql
SELECT werks, rank_in_plant, matnr, maktx, ROUND(inv_value,2) AS inv_value
FROM (
    SELECT
        werks,
        matnr,
        maktx,
        ROUND(labst * unit_price, 2)                                        AS inv_value,
        RANK() OVER (PARTITION BY werks ORDER BY labst * unit_price DESC)   AS rank_in_plant
    FROM materials
    WHERE labst > 0
)
WHERE rank_in_plant <= 3
ORDER BY werks, rank_in_plant;
```

SQLite does **not** allow `WHERE rank_in_plant <= 3` on a window alias in the
same query level — wrapping in a subquery (or CTE) is the correct approach.

</details>


---

## Exercise 5 — Customers Active in Both 2024 and 2025 (Self-Join / Subquery)

**Task:**  
Find all `kunnr` (customers) who placed at least one order (any status) in
**2024** **and** at least one order in **2025**.

Return: `kunnr`, `orders_2024` (count), `orders_2025` (count)  
Sorted by `kunnr` ascending.

Solve using either:
- A **self-join** on `sales_orders`, or
- Two **subqueries** / CTEs joined together, or
- `INTERSECT` (bonus: simplest approach)

*Tests: set logic in SQL, multi-year filtering, advanced join reasoning.*


In [ ]:
%%time

query = """
-- YOUR SQL HERE
-- Approach A (subquery / CTE):
--   Build a set of customers with 2024 orders, another for 2025, INTERSECT or JOIN
-- Approach B (self-join):
--   FROM sales_orders s1 JOIN sales_orders s2 ON s1.kunnr = s2.kunnr
--   WHERE STRFTIME('%Y', s1.order_date) = '2024'
--   AND   STRFTIME('%Y', s2.order_date) = '2025'
-- Approach C (conditional aggregation):
--   GROUP BY kunnr, HAVING COUNT(CASE WHEN year=2024 THEN 1 END) > 0
--         AND COUNT(CASE WHEN year=2025 THEN 1 END) > 0
SELECT 1 AS placeholder
"""

result5 = run_query(query, "Customers Active in Both 2024 and 2025")


<details>
<summary>💡 Hint (click to expand)</summary>

**Conditional aggregation (cleanest for SQLite):**
```sql
SELECT
    kunnr,
    COUNT(CASE WHEN STRFTIME('%Y', order_date) = '2024' THEN 1 END) AS orders_2024,
    COUNT(CASE WHEN STRFTIME('%Y', order_date) = '2025' THEN 1 END) AS orders_2025
FROM sales_orders
GROUP BY kunnr
HAVING orders_2024 > 0 AND orders_2025 > 0
ORDER BY kunnr;
```

**CTE + INTERSECT (most readable):**
```sql
WITH y24 AS (SELECT DISTINCT kunnr FROM sales_orders WHERE STRFTIME('%Y', order_date)='2024'),
     y25 AS (SELECT DISTINCT kunnr FROM sales_orders WHERE STRFTIME('%Y', order_date)='2025')
SELECT kunnr FROM y24
INTERSECT
SELECT kunnr FROM y25
ORDER BY kunnr;
```

</details>


---

## Self-Scoring Rubric

| # | SQL Concept | Solved fluently | Solved with hint | Couldn't solve |
|---|-------------|:---------------:|:----------------:|:--------------:|
| 1 | SELECT, WHERE, ORDER BY, LIMIT | ☐ | ☐ | ☐ |
| 2 | GROUP BY + aggregate functions | ☐ | ☐ | ☐ |
| 3 | CTE + scalar subquery | ☐ | ☐ | ☐ |
| 4 | Window functions (RANK + PARTITION BY) | ☐ | ☐ | ☐ |
| 5 | Set logic / conditional aggregation | ☐ | ☐ | ☐ |

### Interpretation

| Score | Meaning | Suggested next step |
|-------|---------|---------------------|
| 5 × Fluent | SQL is strong | Window functions and CTEs are your key differentiators — lean in |
| 3–4 × Fluent | Solid base | Review CTEs and window functions (RANK, LAG, LEAD, SUM OVER) |
| 1–2 × Fluent | Foundations solid, analytics SQL gaps | Work through Mode Analytics SQL Tutorial sections 3–5 |
| 0 × Fluent | SQL needs structured refresh | Start with SELECT → WHERE → JOIN sequence |

> **Note for analytics/reporting roles:** Exercises 3 and 4 (CTEs + window
> functions) are the most commonly tested in technical interviews for
> analytics engineering and data analyst roles. Prioritise these if
> you're targeting that segment.
